In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import nltk


In [2]:
ratings_df = pd.read_csv('./data/ratings.csv')
books_df = pd.read_csv('./data/books.csv')

In [4]:
print(f"First 5 rows of ratings data:\n{ratings_df.head()}")
print(f"First 5 rows of books data:\n{books_df.head()}")

First 5 rows of ratings data:
   user_id  book_id  rating
0        1      258       5
1        2     4081       4
2        2      260       5
3        2     9296       5
4        2     2318       3
First 5 rows of books data:
   Unnamed: 0  index                            authors  average_rating  \
0           0      0                ['Suzanne Collins']            4.34   
1           1      1  ['J.K. Rowling', 'Mary GrandPré']            4.44   
2           2      2                ['Stephenie Meyer']            3.57   
3           3      3                     ['Harper Lee']            4.25   
4           4      4            ['F. Scott Fitzgerald']            3.89   

   best_book_id  book_id  books_count  \
0       2767052        1          272   
1             3        2          491   
2         41865        3          226   
3          2657        4          487   
4          4671        5         1356   

                                         description  \
0  WINNING MEANS FAM

In [6]:
user_ratings_count = ratings_df['user_id'].value_counts()
book_ratings_count = books_df['book_id'].value_counts()
print(f"Number of ratings per user:\n{user_ratings_count.head()}")
print(f"Number of ratings per book:\n{book_ratings_count.head()}")

Number of ratings per user:
user_id
12874    200
30944    200
52036    199
12381    199
28158    199
Name: count, dtype: int64
Number of ratings per book:
book_id
1       1
7801    1
7793    1
7794    1
7795    1
Name: count, dtype: int64


In [ ]:
def cleanData(df, columns_to_keep, *args):
  clean_books_df = df[columns_to_keep].copy(deep=True)
  clean_books_df['group_text'] = clean_books_df[list(args)].astype(str).agg(" ".join, axis=1)
  clean_books_df = clean_books_df[["book_id", "group_text"]]
  return clean_books_df

cleanData(books_df, ["book_id", "description", "genres", "authors"], "description", "genres", "authors").to_csv('./data/clean_books_df.csv', index=False)
clean_books_df = pd.read_csv('./data/clean_books_df.csv')

In [ ]:
def preprocessText(text):
  lowered_text = text.lower()

  remove_punctuation = re.sub(r'[^\w\s]', '', lowered_text)
  remove_numbers = re.sub(r'[^a-zA-Z\s]', ' ', remove_punctuation)
  clean_text = re.sub(r'\s+', ' ', remove_numbers).strip()

  return clean_text

In [31]:
def build_vectors(clean_books_text):
  vectorizer = TfidfVectorizer(
     stop_words='english', # ignore random english words that don't add meaning
     min_df = 2, # ignore words that appear in less than 2 books
     max_df = 0.9, # ignores words that appear in 90%+ books
     ngram_range=(1,2) # use unigram and bigram
  )
  result =  vectorizer.fit_transform(clean_books_text)
  return vectorizer, result

clean_books_training_text = clean_books_df['group_text'].apply(preprocessText)
vector, result = build_vectors(clean_books_training_text)

print('\nTop 10 most unique words (Highest IDF):')
# Get words and their scores
word_idf_scores = zip(vector.get_feature_names_out(), vector.idf_)

# Sort by IDF score (descending) to see the most unique terms
for word, score in sorted(word_idf_scores, key=lambda x: x[1], reverse=True)[:10]:
    print(f"{word}: {score:.4f}")


Top 10 most unique words (Highest IDF):
aang asks: 9.1118
aang discover: 9.1118
aaron asher: 9.1118
aaron begins: 9.1118
ab: 9.1118
abandon children: 9.1118
abandon city: 9.1118
abandoned airport: 9.1118
abandoned bridge: 9.1118
abandoned family: 9.1118


In [25]:
print('\nWord indexes:')
print(vector.vocabulary_)
print(vector.get_feature_names_out())
print(vector.idf_)


Word indexes:
{'winning': 54268, 'means': 30716, 'fame': 17184, 'fortune': 18591, 'losing': 28885, 'certain': 7845, 'death': 11915, 'hunger': 23084, 'games': 19271, 'begun': 4329, 'ruins': 42268, 'place': 37211, 'known': 26893, 'north': 33956, 'america': 1498, 'lies': 28222, 'nation': 33119, 'panem': 35631, 'shining': 44444, 'capitol': 7191, 'surrounded': 47949, 'outlying': 35142, 'districts': 13658, 'harsh': 21493, 'cruel': 11097, 'keeps': 26285, 'line': 28393, 'forcing': 18411, 'send': 43794, 'boy': 5887, 'girl': 19876, 'ages': 805, 'eighteen': 15085, 'participate': 35881, 'annual': 1894, 'fight': 17740, 'live': 28550, 'tv': 51002, 'sixteen': 45061, 'year': 54906, 'old': 34582, 'katniss': 26175, 'everdeen': 16511, 'regards': 40449, 'sentence': 43852, 'steps': 46869, 'forward': 18598, 'sister': 45020, 'close': 8940, 'dead': 11872, 'survival': 47965, 'second': 43514, 'nature': 33150, 'really': 39979, 'meaning': 30712, 'contender': 10096, 'win': 54205, 'start': 46680, 'making': 29618, 